# core implementation of Step 7.

```
 hemorrhage_diagnosis.csv
          │
          │ read with pandas
          ▼
     DataFrame
          │
          │ load_annotations()
          ▼
       SQLite
          │
          ├── label_taxonomy
          │     → what labels exist
          │
          ├── slice_annotations
          │     → which slice + mask + provenance
          │
          └── slice_labels
                → actual 0/1 label values
```

In concrete terms

The CSV might contain:
```
PatientNumber | SliceNumber | Epidural | Subdural | Fracture_Yes_No
49            | 14          | 1        | 0        | 1
```
We do not simply copy that row into SQLite.

We transform it.

label_taxonomy

Stores the definitions:
```
epidural       → hemorrhage_type
subdural       → hemorrhage_type
fracture       → other_finding
```

slice_annotations

Stores information about the annotation:
```
patient = 049
slice = 14
mask = 049/.../14_HGE_Seg.jpg
source = hemorrhage_diagnosis.csv
source_sha256 = ...
annotator = radiologist-consensus
timestamp = ...
slice_labels
```
Stores the actual assertions:
```
annotation 42 | epidural | 1
annotation 42 | subdural | 0
annotation 42 | fracture | 1
...
```
So the key transformation is:
```
CSV = external/wide representation
             ↓
      normalize + add provenance
             ↓
SQLite = internal/queryable representation
```
And after that, the downstream parts of MedImageForge can query SQLite instead of repeatedly depending on the original CSV.

In [1]:
! python -m medimageforge load-labels

=== Label store loaded ===
Slice annotations: 2501
Label assertions:  15006
With mask:         318
Hemorrhage-positive slices: 318

Label distribution (from the store, not the CSV):
  [hemorrhage_type ] Epidural           173 / 2501
  [hemorrhage_type ] Intraparenchymal    73 / 2501
  [hemorrhage_type ] Intraventricular    24 / 2501
  [hemorrhage_type ] Subarachnoid        18 / 2501
  [hemorrhage_type ] Subdural            56 / 2501
  [other_finding   ] Skull fracture     195 / 2501


## check the 3 tables

In [2]:
from medimageforge.config import load_config, data_path
from medimageforge.manifest import connect

config = load_config()
db = data_path(config, "manifest_db")

with connect(db) as conn:
    tables = conn.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
    """).fetchall()

[t["name"] for t in tables]

['curation',
 'files',
 'label_taxonomy',
 'patients',
 'slice_annotations',
 'slice_labels']

## Check the taxonomy

In [4]:
with connect(db) as conn:
    rows = conn.execute("""
        SELECT code, display_name, category
        FROM label_taxonomy
        ORDER BY category, code
    """).fetchall()

[dict(row) for row in rows]

[{'code': 'epidural',
  'display_name': 'Epidural',
  'category': 'hemorrhage_type'},
 {'code': 'intraparenchymal',
  'display_name': 'Intraparenchymal',
  'category': 'hemorrhage_type'},
 {'code': 'intraventricular',
  'display_name': 'Intraventricular',
  'category': 'hemorrhage_type'},
 {'code': 'subarachnoid',
  'display_name': 'Subarachnoid',
  'category': 'hemorrhage_type'},
 {'code': 'subdural',
  'display_name': 'Subdural',
  'category': 'hemorrhage_type'},
 {'code': 'fracture',
  'display_name': 'Skull fracture',
  'category': 'other_finding'}]

## Check how many annotations we loaded

In [5]:
with connect(db) as conn:
    count = conn.execute("""
        SELECT COUNT(*) AS n
        FROM slice_annotations
    """).fetchone()["n"]

count

2501

## Inspect one real slice

In [6]:
from medimageforge.annotations import get_slice_annotation

record = get_slice_annotation(
    db,
    patient_id="049",
    slice_no=14
)

record

[{'patient_id': '049',
  'slice_no': 14,
  'labels': {'epidural': 1,
   'intraparenchymal': 0,
   'intraventricular': 0,
   'subarachnoid': 0,
   'subdural': 0,
   'fracture': 1},
  'positive_labels': ['epidural', 'fracture'],
  'hemorrhage_types': ['epidural'],
  'no_hemorrhage': False,
  'mask_rel_path': '049/brain/14_mask.png',
  'provenance': {'source_file': 'hemorrhage_diagnosis.csv',
   'source_sha256': 'fb25edc8308411a289d81c0918946c828b7e049c03539b800ed1079a1be3c3dd',
   'annotator': 'radiologist-consensus',
   'annotated_at': '2026-09-15T15:06:44.241759+00:00'}}]